# 16 — Hydro dispatch challenger: storage networks + start/stop economics

**Henrik Remmen (541747), Group 119 — [119] SINTEF Databreach**

Goal: improve on public 0.98990. The reported leader is 0.99562; no hidden-test score is guaranteed.
This notebook is an untrained experiment, not a measured improvement.

New features capture connected storage through tunnels, water budgets in full-load hours,
price cutoffs under those budgets, and a two-state dynamic program accounting for start/stop costs.
The approximate dispatch uses input data only and is not a SHOP reconstruction. Features
retain local water values in EUR/MWh; they do not subtract a downstream water value twice.
Two additional boosting models allow richer interactions. A 50/50 physics ensemble is the control.

Run all cells in a fresh kernel, one notebook at a time. Training A selects rounds and one
global blend weight on 2021, then checks it on 2022. Training B creates CSVs. The last cell
exports standalone final-only notebooks for the professor: no validation reruns, source files,
saved model files or old CSVs are needed. Dependencies are in requirements.txt; no downloads occur.

Resource budget: four CPU threads, sequential fits, float32 features, resumable unit checkpoints.
Development entails 112 unit fits and up to 56 final fits; early stopping bounds new-model work.
Final-only reproduction requires at most 56 unit fits. The observed earlier 14-model XGBoost
run took 7.7 minutes on this Mac at four threads; the larger new models are unbenchmarked.
Allow hours for the development run. The 12-hour requirement applies per final short notebook;
confirm cold-start runtime on four cores before submission. Peak RAM is not hard-capped.

All years have informed prior project work; 2022 is a historical audit, not a pristine holdout.
The old controls also use previously selected constants. The development score is optimistic
because it selects boosting rounds. Public Kaggle scores never enter the selection code.
If the proposed mixture fails the audit, the recommended CSV reverts to the baseline and a
separate experimental CSV remains available. The notebook reports this explicitly.

In [1]:
from pathlib import Path
import os
import time
from IPython.display import display

NOTEBOOK_STARTED = time.perf_counter()
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/kernel').is_dir() and (p / 'prediction_mapping.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Place notebook beside the supplied data/ directory and mapping files.')
THREADS = min(4, max(1, os.cpu_count() or 1))
for key in ('OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[key] = '1'
os.environ['OMP_NUM_THREADS'] = str(THREADS)
SOURCE_DIGEST = '98bcf13847939c54db9f70d894639de3ad3cc72457e7ccb4281150e730f2ee82'

# Existing models are frozen controls. Their counts were determined in prior development.
BASELINE_ROUNDS = {
 'xgb_v2': dict(zip(
  ['Hogga_G1','Tokke_G1','Tokke_G2','Tokke_G3','Tokke_G4','Lio_G1','Byrte_G1',
   'Vinje_G1','Vinje_G2','Vinje_G3','Haukeli_G1','Songa_G1','Kjela_G1','Vesle Kjela_G1'],
  [410,450,854,587,693,249,292,658,404,607,223,510,326,372])),
 'lgb_v2': dict(zip(
  ['Hogga_G1','Tokke_G1','Tokke_G2','Tokke_G3','Tokke_G4','Lio_G1','Byrte_G1',
   'Vinje_G1','Vinje_G2','Vinje_G3','Haukeli_G1','Songa_G1','Kjela_G1','Vesle Kjela_G1'],
  [398,534,387,458,410,386,441,852,299,873,330,465,393,177]))}
xgb_parameters = dict(objective='binary:logistic', eval_metric='auc', learning_rate=.03,
    max_depth=7, min_child_weight=50, subsample=.85, colsample_bytree=.85,
    reg_alpha=.1, reg_lambda=5., tree_method='hist', max_bin=256, device='cpu', random_state=42)
lgb_parameters = dict(objective='binary', metric='None', learning_rate=.03, num_leaves=31,
    max_depth=-1, min_data_in_leaf=150, feature_fraction=.85, bagging_fraction=.85,
    bagging_freq=1, lambda_l1=.1, lambda_l2=5., max_bin=255, device_type='cpu',
    deterministic=True, force_col_wise=True, seed=42, verbosity=-1)
SPECS = {
 'xgb_v2': dict(backend='xgboost', features='physics', max_rounds=2000, parameters=xgb_parameters),
 'lgb_v2': dict(backend='lightgbm', features='physics', max_rounds=1800, parameters=lgb_parameters),
 'xgb_dispatch': dict(backend='xgboost', features='dispatch', max_rounds=1400,
    parameters=dict(xgb_parameters, learning_rate=.04, max_depth=8, min_child_weight=20,
                    reg_lambda=8., colsample_bytree=.9)),
 'lgb_dispatch': dict(backend='lightgbm', features='dispatch', max_rounds=1400,
    parameters=dict(lgb_parameters, learning_rate=.04, num_leaves=63, min_data_in_leaf=100,
                    lambda_l2=8., feature_fraction=.9)),
}
print(f'At most {THREADS} CPU threads, one fit at a time. Training has not started.')

At most 4 CPU threads, one fit at a time. Training has not started.


## Definitions: pipeline (no training)

In [2]:
"""Shared, target-safe data and feature pipeline used by experiment notebooks.

The module contains no model training.  It exists so CatBoost and XGBoost
experiments use exactly the same inputs, folds, metric, and submission layout.
"""

from dataclasses import dataclass
from pathlib import Path
import re

import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import roc_auc_score


HORIZON = 168
RANDOM_SEED = 42


@dataclass
class ProjectData:
    root: Path
    X_all: pd.DataFrame
    target_cube: np.ndarray
    units: list[str]
    train_cases: pd.DataFrame
    test_cases: pd.DataFrame
    sample_submission: pd.DataFrame
    feature_sets: dict[str, list[str]]
    feature_group: dict[str, str]
    generator_features: pd.DataFrame
    folds: list[dict]

    @property
    def n_train_cases(self):
        return len(self.train_cases)

    @property
    def n_test_cases(self):
        return len(self.test_cases)

    @property
    def n_all_cases(self):
        return self.n_train_cases + self.n_test_cases


def read_time_csv(path, time_column="date"):
    frame = pd.read_csv(path)
    frame[time_column] = pd.to_datetime(frame[time_column], utc=True)
    return frame


def rows_for_cases(case_ids):
    case_ids = np.asarray(case_ids)
    return (
        case_ids[:, None] * HORIZON
        + np.arange(HORIZON)[None, :]
    ).ravel()


def micro_auc(y_true_cube, y_pred_cube):
    if y_true_cube.shape != y_pred_cube.shape:
        raise ValueError("Truth and prediction cubes must have identical shapes")
    return roc_auc_score(
        y_true_cube.reshape(len(y_true_cube), -1),
        y_pred_cube.reshape(len(y_pred_cube), -1),
        average="micro",
    )


def per_unit_auc(y_true_cube, y_pred_cube, units):
    return {
        unit: roc_auc_score(
            y_true_cube[:, unit_index, :].ravel(),
            y_pred_cube[:, unit_index, :].ravel(),
        )
        for unit_index, unit in enumerate(units)
    }


def make_purged_folds(train_cases, validation_years=(2020, 2021, 2022)):
    starts = pd.DatetimeIndex(train_cases["starttime"])
    folds = []
    for year in validation_years:
        validation_start = pd.Timestamp(f"{year}-01-01", tz="UTC")
        validation_end = pd.Timestamp(f"{year + 1}-01-01", tz="UTC")
        validation_cases = np.flatnonzero(
            (starts >= validation_start) & (starts < validation_end)
        )
        training_cases = np.flatnonzero(
            starts + pd.Timedelta(hours=HORIZON - 1) < validation_start
        )
        if not len(training_cases) or not len(validation_cases):
            raise ValueError(f"Empty training or validation partition for {year}")
        if not (
            starts[training_cases].max() + pd.Timedelta(hours=HORIZON - 1)
            < starts[validation_cases].min()
        ):
            raise AssertionError(f"Fold {year} is not temporally purged")
        folds.append({
            "year": year,
            "train_cases": training_cases,
            "validation_cases": validation_cases,
        })
    return folds


def make_submission(data, prediction_cube, filename):
    expected = (data.n_test_cases, len(data.units), HORIZON)
    if prediction_cube.shape != expected:
        raise ValueError(f"Expected predictions {expected}, got {prediction_cube.shape}")

    submission_data = {"Run No": data.test_cases["Run No"].to_numpy()}
    for unit_index, unit in enumerate(data.units):
        for horizon in range(HORIZON):
            submission_data[f"result_committed_{unit}_t{horizon}"] = (
                prediction_cube[:, unit_index, horizon]
            )

    submission = pd.DataFrame(submission_data)[data.sample_submission.columns]
    probabilities = submission.iloc[:, 1:].to_numpy()
    if not submission["Run No"].equals(data.sample_submission["Run No"]):
        raise AssertionError("Run No does not match sample_submission.csv")
    if not np.isfinite(probabilities).all():
        raise ValueError("Submission contains missing or non-finite predictions")
    if probabilities.min() < 0 or probabilities.max() > 1:
        raise ValueError("Submission probabilities are outside [0, 1]")

    path = data.root / "submissions" / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(path, index=False)
    return path, submission


def build_project_data(root):
    root = Path(root).resolve()
    data_dir = root / "data"

    price = read_time_csv(data_dir / "kernel" / "Historical_day_ahead_price_2015_2025.csv")
    inflow = read_time_csv(data_dir / "kernel" / "Historical_inflow_1958_2025.csv")
    volume = read_time_csv(data_dir / "kernel" / "Historical_volume_2015_2024.csv")
    water_value = read_time_csv(data_dir / "kernel" / "Synthetic_water_value_2015_2024.csv")
    min_flow = read_time_csv(data_dir / "extended" / "Constraint_min_flow.csv")
    min_volume = read_time_csv(data_dir / "extended" / "Constraint_min_volume.csv")
    targets = read_time_csv(data_dir / "kernel" / "Unit_commitment_decisions.csv", "starttime")
    prediction_mapping = read_time_csv(root / "prediction_mapping.csv", "starttime")
    sample_submission = pd.read_csv(root / "sample_submission.csv")

    with open(data_dir / "extended" / "Tokke_Vinje_topology.yaml") as stream:
        topology = yaml.safe_load(stream)

    pattern = re.compile(r"^result_committed_(.+)_t(\d+)$")
    target_columns = [column for column in targets if pattern.match(column)]
    units = list(dict.fromkeys(pattern.match(column).group(1) for column in target_columns))
    if len(units) != 14 or len(target_columns) != 14 * HORIZON:
        raise AssertionError("Expected 14 units and 168 targets per unit")
    if target_columns != list(sample_submission.columns[1:]):
        raise AssertionError("Target order differs from sample submission")

    target_cube = np.stack([
        targets[[f"result_committed_{unit}_t{hour}" for hour in range(HORIZON)]]
        .to_numpy(dtype=np.int8)
        for unit in units
    ], axis=1)
    if not set(np.unique(target_cube)).issubset({0, 1}):
        raise AssertionError("Targets must be binary")

    train_cases = targets[["Run No", "starttime"]].copy()
    test_cases = prediction_mapping[["Run No", "starttime"]].copy()
    all_cases = pd.concat([train_cases, test_cases], ignore_index=True)
    n_all_cases = len(all_cases)
    starts = pd.DatetimeIndex(all_cases["starttime"])
    horizon_hour = np.tile(np.arange(HORIZON), n_all_cases)
    target_time = pd.DatetimeIndex(
        starts.repeat(HORIZON) + pd.to_timedelta(horizon_hour, unit="h")
    )

    def numeric_indexed(frame):
        return frame.set_index("date").select_dtypes(include=np.number)

    def hourly_matrix(series):
        values = series.reindex(target_time).to_numpy()
        matrix = values.reshape(n_all_cases, HORIZON)
        if not np.isfinite(matrix).all():
            raise ValueError(
                f"Missing/non-finite hourly inputs: {(~np.isfinite(matrix)).sum()}"
            )
        return matrix

    def repeat_case(values):
        values = np.asarray(values)
        if len(values) != n_all_cases:
            raise ValueError("Case-level array has incorrect length")
        return np.repeat(values, HORIZON)

    def cyclic(values, period):
        angle = 2 * np.pi * np.asarray(values) / period
        return np.sin(angle), np.cos(angle)

    def clipped_shift(matrix, offset):
        indices = np.clip(np.arange(HORIZON) + offset, 0, HORIZON - 1)
        return matrix[:, indices]

    def centred_rolling_mean(matrix, window):
        return (
            pd.DataFrame(matrix).T
            .rolling(window=window, min_periods=1, center=True)
            .mean().T.to_numpy()
        )

    feature_data = {}
    feature_group = {}

    def add_feature(name, values, group):
        values = np.asarray(values)
        expected = n_all_cases * HORIZON
        if values.ndim != 1 or len(values) != expected:
            raise ValueError(f"{name}: expected {expected}, got {values.shape}")
        feature_data[name] = values.astype(np.float32)
        feature_group[name] = group

    hour = target_time.hour.to_numpy()
    weekday = target_time.dayofweek.to_numpy()
    month = target_time.month.to_numpy()
    for name, values, period in (
        ("hour", hour, 24),
        ("weekday", weekday, 7),
        ("month", month - 1, 12),
    ):
        sine, cosine = cyclic(values, period)
        add_feature(f"{name}_sin", sine, "calendar")
        add_feature(f"{name}_cos", cosine, "calendar")
    add_feature("horizon_hour", horizon_hour, "calendar")

    price_matrix = hourly_matrix(
        price.set_index("date")["Day-ahead price (EUR/MWh)"]
    )
    price_mean = price_matrix.mean(axis=1)
    price_std = price_matrix.std(axis=1)
    price_rank = pd.DataFrame(price_matrix).rank(axis=1, pct=True).to_numpy()
    add_feature("price", price_matrix.ravel(), "price")
    add_feature("price_week_mean", repeat_case(price_mean), "price")
    add_feature("price_week_std", repeat_case(price_std), "price")
    add_feature("price_week_percentile", price_rank.ravel(), "price")
    add_feature("price_minus_week_mean", (price_matrix - price_mean[:, None]).ravel(), "price")
    add_feature(
        "price_week_zscore",
        ((price_matrix - price_mean[:, None]) / (price_std[:, None] + 1e-6)).ravel(),
        "price_windows",
    )
    add_feature("price_top_10pct", (price_rank >= 0.90).ravel(), "price_windows")
    add_feature("price_bottom_10pct", (price_rank <= 0.10).ravel(), "price_windows")
    for offset in (1, 3, 6, 24):
        lag = clipped_shift(price_matrix, -offset)
        lead = clipped_shift(price_matrix, offset)
        add_feature(f"price_lag_{offset}h", lag.ravel(), "price_windows")
        add_feature(f"price_lead_{offset}h", lead.ravel(), "price_windows")
        add_feature(f"price_change_from_{offset}h", (price_matrix - lag).ravel(), "price_windows")
        add_feature(f"price_change_to_next_{offset}h", (lead - price_matrix).ravel(), "price_windows")
    for window in (3, 6, 12, 24, 48):
        rolling = centred_rolling_mean(price_matrix, window)
        add_feature(f"price_mean_{window}h", rolling.ravel(), "price_windows")
        add_feature(f"price_minus_mean_{window}h", (price_matrix - rolling).ravel(), "price_windows")

    reservoirs = [column for column in volume.columns if column != "date"]
    capacity = {
        name: values["max_vol"]
        for name, values in topology["model"]["reservoir"].items()
    }
    start_volume = numeric_indexed(volume).reindex(starts)[reservoirs]
    start_water = numeric_indexed(water_value).reindex(starts)[reservoirs]
    end_water = numeric_indexed(water_value).reindex(
        starts + pd.Timedelta(days=7)
    )[reservoirs]
    if start_volume.isna().any().any() or start_water.isna().any().any() or end_water.isna().any().any():
        raise ValueError("Missing case-level reservoir data")

    for reservoir in reservoirs:
        fill = start_volume[reservoir].to_numpy() / capacity[reservoir]
        water_start = start_water[reservoir].to_numpy()
        water_end = end_water[reservoir].to_numpy()
        add_feature(f"fill__{reservoir}", repeat_case(fill), "hydrology")
        add_feature(f"water_value_end__{reservoir}", repeat_case(water_end), "water_margins")
        add_feature(f"water_value_change__{reservoir}", repeat_case(water_end - water_start), "water_margins")
        add_feature(f"price_margin__{reservoir}", (price_matrix - water_end[:, None]).ravel(), "water_margins")

    system_fill = (
        start_volume.sum(axis=1).to_numpy()
        / sum(capacity[reservoir] for reservoir in reservoirs)
    )
    water_end_mean = end_water.mean(axis=1).to_numpy()
    add_feature("system_filling_ratio", repeat_case(system_fill), "hydrology")
    add_feature("water_value_end_mean", repeat_case(water_end_mean), "water_margins")
    margin = price_matrix - water_end_mean[:, None]
    add_feature("price_minus_water_value_mean", margin.ravel(), "water_margins")
    add_feature(
        "price_to_abs_water_value_ratio",
        (price_matrix / (np.abs(water_end_mean[:, None]) + 1.0)).ravel(),
        "water_margins",
    )

    inflow_values = numeric_indexed(inflow)
    inflow_matrices = {}
    for column in inflow_values:
        matrix = hourly_matrix(inflow_values[column])
        inflow_matrices[column] = matrix
        add_feature(f"inflow__{column}", matrix.ravel(), "hydrology")
        add_feature(f"inflow_week_mean__{column}", repeat_case(matrix.mean(axis=1)), "hydrology")
    total_inflow = np.sum(list(inflow_matrices.values()), axis=0)
    add_feature("inflow_total", total_inflow.ravel(), "hydrology")
    add_feature("inflow_total_week_mean", repeat_case(total_inflow.mean(axis=1)), "hydrology")
    add_feature("inflow_total_week_std", repeat_case(total_inflow.std(axis=1)), "hydrology")
    add_feature("inflow_total_24h_mean", centred_rolling_mean(total_inflow, 24).ravel(), "hydrology")
    add_feature(
        "inflow_total_cumulative_mean",
        (np.cumsum(total_inflow, axis=1) / (np.arange(HORIZON)[None, :] + 1)).ravel(),
        "hydrology",
    )

    flow_values = numeric_indexed(min_flow)
    min_volume_values = numeric_indexed(min_volume)
    flow_matrices = {}
    min_volume_matrices = {}
    for column in flow_values:
        matrix = hourly_matrix(flow_values[column])
        flow_matrices[column] = matrix
        add_feature(f"min_flow__{column}", matrix.ravel(), "constraints")
    for column in min_volume_values:
        matrix = hourly_matrix(min_volume_values[column])
        min_volume_matrices[column] = matrix
        add_feature(f"min_volume__{column}", matrix.ravel(), "constraints")
        if column in reservoirs:
            initial = start_volume[column].to_numpy()[:, None]
            add_feature(
                f"min_volume_slack__{column}",
                ((initial - matrix) / capacity[column]).ravel(),
                "constraints",
            )

    flow_active_count = np.sum([matrix > 0 for matrix in flow_matrices.values()], axis=0)
    volume_active_count = np.sum([matrix > 0 for matrix in min_volume_matrices.values()], axis=0)
    total_min_flow = np.sum(list(flow_matrices.values()), axis=0)
    total_min_volume = np.sum(list(min_volume_matrices.values()), axis=0)
    add_feature("min_flow_active_count", flow_active_count.ravel(), "constraints")
    add_feature("min_volume_active_count", volume_active_count.ravel(), "constraints")
    add_feature("min_flow_total", total_min_flow.ravel(), "constraints")
    add_feature("min_volume_total", total_min_volume.ravel(), "constraints")

    variable_flow_columns = [
        column for column in flow_values
        if 0 < (flow_values[column] > 0).mean() < 1
    ]
    variable_volume_columns = [
        column for column in min_volume_values
        if 0 < (min_volume_values[column] > 0).mean() < 1
    ]
    variable_active = np.zeros((n_all_cases, HORIZON), dtype=bool)
    for column in variable_flow_columns:
        variable_active |= flow_matrices[column] > 0
    for column in variable_volume_columns:
        variable_active |= min_volume_matrices[column] > 0
    add_feature("variable_constraint_active", variable_active.ravel(), "constraints")

    add_feature("margin_x_system_fill", (margin * system_fill[:, None]).ravel(), "interactions")
    add_feature("price_rank_x_system_fill", (price_rank * system_fill[:, None]).ravel(), "interactions")
    add_feature("margin_x_total_inflow", (margin * total_inflow).ravel(), "interactions")
    add_feature("price_rank_x_hour_sin", price_rank.ravel() * feature_data["hour_sin"], "interactions")
    add_feature("price_rank_x_hour_cos", price_rank.ravel() * feature_data["hour_cos"], "interactions")
    add_feature("margin_x_constraint", margin.ravel() * variable_active.ravel(), "interactions")

    X_all = pd.DataFrame(feature_data)
    constants = [
        column for column in X_all
        if X_all[column].nunique(dropna=False) <= 1
    ]
    X_all = X_all.drop(columns=constants)
    if X_all.isna().any().any() or not np.isfinite(X_all.to_numpy()).all():
        raise ValueError("Final feature matrix contains invalid values")

    baseline = [
        "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
        "month_sin", "month_cos", "horizon_hour", "price",
        "price_week_mean", "price_week_std", "price_week_percentile",
        "price_minus_week_mean", "water_value_end_mean",
        "price_minus_water_value_mean", "system_filling_ratio",
        "inflow_total", "inflow_total_24h_mean", "min_flow_total",
        "min_volume_total", "variable_constraint_active",
    ]

    def grouped(name):
        return [
            feature for feature, group in feature_group.items()
            if group == name and feature in X_all.columns
        ]

    def unique_existing(names):
        return list(dict.fromkeys(name for name in names if name in X_all.columns))

    feature_sets = {
        "baseline": unique_existing(baseline),
        "baseline_plus_price": unique_existing(baseline + grouped("price_windows")),
        "baseline_plus_price_water": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
        ),
        "baseline_plus_price_water_hydrology": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
            + grouped("hydrology")
        ),
        "full_engineered": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
            + grouped("hydrology") + grouped("constraints")
            + grouped("interactions")
        ),
    }

    generator_rows = []
    generator_config = topology["model"]["generator"]
    for unit in units:
        config = generator_config[unit]
        plant = unit.rsplit("_G", 1)[0]
        number_match = re.search(r"_G(\d+)$", unit)
        generator_rows.append({
            "unit": unit,
            "plant": plant,
            "generator_number": int(number_match.group(1)) if number_match else 1,
            "p_min": config.get("p_min", np.nan),
            "p_max": config.get("p_max", np.nan),
            "p_nom": config.get("p_nom", np.nan),
            "minimum_output_fraction": config.get("p_min", np.nan) / config.get("p_max", np.nan),
            "start_cost": config.get("startcost_const", 0.0),
            "stop_cost": config.get("stopcost_const", 0.0),
            "penstock": str(config.get("penstock", "unknown")),
        })
    generator_features = pd.DataFrame(generator_rows)
    generator_features["units_in_plant"] = (
        generator_features.groupby("plant")["unit"].transform("count")
    )

    return ProjectData(
        root=root,
        X_all=X_all,
        target_cube=target_cube,
        units=units,
        train_cases=train_cases,
        test_cases=test_cases,
        sample_submission=sample_submission,
        feature_sets=feature_sets,
        feature_group=feature_group,
        generator_features=generator_features,
        folds=make_purged_folds(train_cases),
    )

## Definitions: physics (no training)

In [3]:
"""Deterministic input-only features for the next hydropower experiments.

No fitted transforms, labels, predicted/observed future storage, or solver outputs.
Water budgets deliberately exclude unknown dispatch and are explicitly proxies.
"""

import numpy as np
import pandas as pd
import yaml



def window_sum(values, width, forward=True):
    """Truncated within-case sum, including the current hour; no edge repeats."""
    if not forward:
        return window_sum(values[:, ::-1], width, True)[:, ::-1]
    cumulative = np.pad(np.cumsum(values, axis=1, dtype=np.float64), ((0, 0), (1, 0)))
    left = np.arange(values.shape[1])
    right = np.minimum(left + width, values.shape[1])
    return cumulative[:, right] - cumulative[:, left]


def consecutive_true(mask):
    """Run length ending/starting at each position, reset at each case boundary."""
    past = np.zeros(mask.shape, dtype=np.float32)
    future = np.zeros_like(past)
    for h in range(mask.shape[1]):
        past[:, h] = mask[:, h] * (1 + (past[:, h - 1] if h else 0))
    for h in range(mask.shape[1] - 1, -1, -1):
        future[:, h] = mask[:, h] * (1 + (future[:, h + 1] if h + 1 < mask.shape[1] else 0))
    return past, future


def plant_sources(topology, plant):
    """Follow reservoir -> plant or reservoir -> tunnel -> plant connections.

    Stop at the first reservoir; do not pretend all upstream water arrives instantly.
    """
    connections = topology['connections']
    found = set()

    def visit(name, kind, seen):
        if (name, kind) in seen:
            return
        seen = seen | {(name, kind)}
        for edge in connections:
            if edge['to'] != name or edge['to_type'] != kind:
                continue
            if edge['from_type'] == 'reservoir':
                found.add(edge['from'])
            elif edge['from_type'] == 'tunnel':
                visit(edge['from'], 'tunnel', seen)

    visit(plant, 'plant', set())
    if not found:
        raise ValueError(f'No source reservoir found for {plant}')
    return sorted(found)


def build_research_features(data):
    """Return global additions, per-plant additions, and an auditable source mapping."""
    root = data.root
    with open(root / 'data/extended/Tokke_Vinje_topology.yaml') as stream:
        topology = yaml.safe_load(stream)
    starts = pd.DatetimeIndex(pd.concat([data.train_cases, data.test_cases])['starttime'])
    n = data.n_all_cases
    horizon = np.arange(HORIZON)
    time_index = starts.repeat(HORIZON) + pd.to_timedelta(np.tile(horizon, n), unit='h')
    price = data.X_all['price'].to_numpy().reshape(n, HORIZON)
    volume = read_time_csv(root / 'data/kernel/Historical_volume_2015_2024.csv').set_index('date').reindex(starts)
    water = read_time_csv(root / 'data/kernel/Synthetic_water_value_2015_2024.csv').set_index('date').reindex(starts + pd.Timedelta(days=7))
    inflow = read_time_csv(root / 'data/kernel/Historical_inflow_1958_2025.csv').set_index('date').reindex(time_index)
    minimum = read_time_csv(root / 'data/extended/Constraint_min_volume.csv').set_index('date').reindex(time_index)
    extra = {}

    def add(target, name, values):
        values = np.broadcast_to(values, (n, HORIZON)).reshape(-1).astype(np.float32)
        if not np.isfinite(values).all():
            raise ValueError(f'Non-finite research feature {name}')
        target[name] = values

    add(extra, 'v2_day_in_horizon', np.broadcast_to(horizon // 24, (n, HORIZON)))
    add(extra, 'v2_hours_remaining', HORIZON - horizon)
    day_price = price.reshape(n, 7, 24)
    day_mean = np.repeat(day_price.mean(axis=2), 24, axis=1)
    day_rank = pd.DataFrame(day_price.reshape(-1, 24)).rank(axis=1, pct=True).to_numpy().reshape(n, HORIZON)
    add(extra, 'v2_price_day_rank', day_rank)
    add(extra, 'v2_price_minus_day_mean', price - day_mean)
    prefix_mean = np.cumsum(price, axis=1) / (horizon + 1)
    suffix_mean = np.cumsum(price[:, ::-1], axis=1)[:, ::-1] / (HORIZON - horizon)
    add(extra, 'v2_price_minus_prefix_mean', price - prefix_mean)
    add(extra, 'v2_price_minus_suffix_mean', price - suffix_mean)
    add(extra, 'v2_best_remaining_price_gap', np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1] - price)
    reservoirs = topology['model']['reservoir']
    reservoir_inputs = {}
    for name, config in reservoirs.items():
        cap = float(config['max_vol'])
        initial = volume[name].to_numpy()[:, None]
        terminal = water[name].to_numpy()[:, None]
        flow = inflow[name].to_numpy().reshape(n, HORIZON)
        # m3/s * 3600 / 1e6 -> million m3 per hour, matching storage units.
        hourly_volume = flow * 0.0036
        cumulative = np.cumsum(hourly_volume, axis=1)
        remaining = np.cumsum(hourly_volume[:, ::-1], axis=1)[:, ::-1]
        floor = minimum[name].to_numpy().reshape(n, HORIZON) if name in minimum else np.zeros_like(price)
        budget = initial + cumulative
        add(extra, f'v2_cumulative_inflow_fill__{name}', cumulative / cap)
        add(extra, f'v2_remaining_inflow_fill__{name}', remaining / cap)
        add(extra, f'v2_no_release_excess__{name}', np.maximum(budget - cap, 0) / cap)
        add(extra, f'v2_no_release_min_slack__{name}', (budget - floor) / cap)
        curve = config['vol_head']
        head = np.interp(initial[:, 0], curve['x'], curve['y'])[:, None]
        reservoir_inputs[name] = (cap, initial, terminal, cumulative, remaining, head)

    plants = {}
    mappings = []
    for plant, plant_config in topology['model']['plant'].items():
        sources = plant_sources(topology, plant)
        values = [reservoir_inputs[s] for s in sources]
        capacity = sum(v[0] for v in values)
        initial = sum(v[1] for v in values)
        # Multi-reservoir summaries are capacity-weighted approximations, not SHOP prices.
        terminal = sum(v[2] * v[0] for v in values) / capacity
        head = sum(v[5] * v[0] for v in values) / capacity - float(plant_config['outlet_line'])
        cum = sum(v[3] for v in values)
        remain = sum(v[4] for v in values)
        margin = price - terminal
        out = {}
        add(out, 'local_initial_fill', initial / capacity)
        add(out, 'local_gross_head_proxy', head)
        add(out, 'local_terminal_water_value', terminal)
        add(out, 'local_margin', margin)
        add(out, 'local_margin_normalized', margin / (np.abs(terminal) + price.std(axis=1)[:, None] + 1))
        add(out, 'local_cumulative_inflow_fill', cum / capacity)
        add(out, 'local_remaining_inflow_fill', remain / capacity)
        add(out, 'local_margin_x_fill', margin * initial / capacity)
        positive = np.maximum(margin, 0)
        negative = np.maximum(-margin, 0)
        past, future = consecutive_true(margin > 0)
        add(out, 'local_profitable_run_past', past)
        add(out, 'local_profitable_run_future', future)
        add(out, 'local_profitable_fraction_week', (margin > 0).mean(axis=1)[:, None])
        for hours in (6, 24):
            add(out, f'local_positive_margin_next_{hours}h', window_sum(positive, hours))
            add(out, f'local_negative_margin_next_{hours}h', window_sum(negative, hours))
            add(out, f'local_positive_margin_past_{hours}h', window_sum(positive, hours, False))
        # Start cost / rated production gives EUR/MWh; margin sums are hourly
        # EUR/MWh contributions. This is an approximate rated-power payback ratio.
        unit_configs = [topology['model']['generator'][u] for u in data.units if u.rsplit('_G', 1)[0] == plant]
        cost_per_mw = np.mean([(v.get('startcost_const', 0) + v.get('stopcost_const', 0)) / v['p_max'] for v in unit_configs])
        add(out, 'local_start_stop_payback_proxy', window_sum(positive, 24) / max(cost_per_mw, 1e-6))
        plants[plant] = pd.DataFrame(out)
        mappings.append({'plant': plant, 'source_reservoirs': '|'.join(sources), 'source_capacity_Mm3': capacity})
    return pd.DataFrame(extra), plants, pd.DataFrame(mappings)

## Definitions: dispatch (no training)

In [4]:
"""Input-only storage-network and commitment features; never fits a model.

The dispatch calculation is a two-state economic proxy, not a SHOP simulation.
It uses the supplied 168-hour price/inflow path, initial volume and terminal
water values. No future observed volumes or commitment labels are consulted.
"""
import numpy as np
import pandas as pd
import yaml


def commitment_advantage(reward, start_cost, stop_cost):
    """Best objective with hour ON minus best objective with hour OFF.

    Exhaustive two-state dynamic program with a free initial and terminal state.
    Reward and costs have the same units. Input shape is cases x horizon.
    """
    reward = np.asarray(reward, dtype=np.float64)
    n, h = reward.shape
    f0 = np.zeros((n, h)); f1 = np.zeros_like(f0)
    a = np.zeros(n); b = np.zeros(n)
    for t in range(h):
        a, b = np.maximum(a, b - stop_cost), np.maximum(b, a - start_cost) + reward[:, t]
        f0[:, t], f1[:, t] = a, b
    a = np.zeros(n); b = np.zeros(n)
    result = np.empty_like(reward)
    for t in range(h - 1, -1, -1):
        result[:, t] = f1[:, t] + b - f0[:, t] - a
        a, b = (np.maximum(a, b + reward[:, t] - start_cost),
                np.maximum(a - stop_cost, b + reward[:, t]))
    return result.astype(np.float32)


def tunnel_storage_group(topology, sources):
    """Include upstream reservoirs connected without passing through a turbine.

    This storage group is an availability proxy: tunnel limits still matter.
    """
    found = set(sources)
    pending = [(s, 'reservoir') for s in sources]
    seen = set()
    while pending:
        node = pending.pop()
        if node in seen:
            continue
        seen.add(node)
        for edge in topology['connections']:
            if (edge['to'], edge['to_type']) != node:
                continue
            if edge['from_type'] == 'tunnel':
                pending.append((edge['from'], 'tunnel'))
            elif edge['from_type'] == 'reservoir' and node[1] == 'tunnel':
                found.add(edge['from'])
                pending.append((edge['from'], 'reservoir'))
    return sorted(found)


def build_challenger_features(data):
    with open(data.root / 'data/extended/Tokke_Vinje_topology.yaml') as stream:
        topology = yaml.safe_load(stream)
    n = data.n_all_cases
    h = np.arange(HORIZON)
    starts = pd.DatetimeIndex(pd.concat([data.train_cases, data.test_cases])['starttime'])
    times = starts.repeat(HORIZON) + pd.to_timedelta(np.tile(h, n), unit='h')
    price = data.X_all['price'].to_numpy().reshape(n, HORIZON)
    volume = read_time_csv(data.root / 'data/kernel/Historical_volume_2015_2024.csv').set_index('date').reindex(starts)
    water = read_time_csv(data.root / 'data/kernel/Synthetic_water_value_2015_2024.csv').set_index('date').reindex(starts + pd.Timedelta(days=7))
    inflow = read_time_csv(data.root / 'data/kernel/Historical_inflow_1958_2025.csv').set_index('date').reindex(times)
    floors = read_time_csv(data.root / 'data/extended/Constraint_min_volume.csv').set_index('date').reindex(times)
    min_flow = read_time_csv(data.root / 'data/extended/Constraint_min_flow.csv').set_index('date').reindex(times)
    reservoirs = topology['model']['reservoir']
    plants, audit = {}, []
    for plant, cfg in topology['model']['plant'].items():
        sources = plant_sources(topology, plant)
        group = tunnel_storage_group(topology, sources)
        # Broader passive catchment, kept distinct from tunnel-connected storage.
        # In particular Langeidvatn feeds Vatjern/Haukeli through a river, not a tunnel.
        seen, pending = set(), [(s, 'reservoir') for s in sources]
        while pending:
            node = pending.pop()
            if node in seen:
                continue
            seen.add(node)
            pending.extend((e['from'], e['from_type']) for e in topology['connections']
                           if (e['to'], e['to_type']) == node
                           and e['from_type'] in ('reservoir', 'river', 'tunnel'))
        catchment = sorted(name for name, kind in seen if kind == 'reservoir')
        rivers = sorted(name for name, kind in seen if kind == 'river')
        downstream = sorted({e['to'] for e in topology['connections']
                             if e['from'] == plant and e['from_type'] == 'plant'
                             and e['to_type'] == 'reservoir'})
        unit_configs = [v for u, v in topology['model']['generator'].items()
                        if u.rsplit('_G', 1)[0] == plant]
        pmax = sum(float(v['p_max']) for v in unit_configs)
        pmin_fraction = sum(float(v['p_min']) for v in unit_configs) / pmax
        startup = sum(float(v['startcost_const']) for v in unit_configs) / pmax
        shutdown = sum(float(v['stopcost_const']) for v in unit_configs) / pmax
        source_capacity = sum(float(reservoirs[s]['max_vol']) for s in sources)
        cap = sum(float(reservoirs[s]['max_vol']) for s in group)
        initial = volume[group].sum(axis=1).to_numpy()[:, None]
        source_initial = volume[sources].sum(axis=1).to_numpy()[:, None]
        flow = inflow[group].sum(axis=1).to_numpy().reshape(n, HORIZON)
        minimum = sum((floors[s].to_numpy().reshape(n, HORIZON) for s in group if s in floors), np.zeros_like(price))
        terminal = sum(water[s].to_numpy()[:, None] * reservoirs[s]['max_vol'] for s in sources) / source_capacity
        head = sum(np.interp(volume[s], reservoirs[s]['vol_head']['x'], reservoirs[s]['vol_head']['y'])[:, None]
                   * reservoirs[s]['max_vol'] for s in sources) / source_capacity - float(cfg['outlet_line'])
        # p[MW] = 0.00981 * head[m] * flow[m3/s] * efficiency.
        # 90% efficiency is an explicit approximation, not a measured turbine curve.
        rated_flow = pmax / (0.00981 * np.maximum(head, 1.) * 0.90)
        hourly_release = rated_flow * 0.0036
        incoming = flow * 0.0036
        cumulative = np.cumsum(incoming, axis=1)
        remaining = np.cumsum(incoming[:, ::-1], axis=1)[:, ::-1]
        available = np.maximum(initial + incoming.sum(axis=1)[:, None] - minimum.max(axis=1)[:, None], 0)
        hours_budget = np.clip(available / hourly_release, 0, HORIZON)
        required_hours = np.clip((initial + incoming.sum(axis=1)[:, None] - cap) / hourly_release, 0, HORIZON)
        sorted_price = np.sort(price, axis=1)[:, ::-1]
        cutoff_index = np.clip(np.ceil(hours_budget[:, 0]).astype(int) - 1, 0, HORIZON - 1)
        scarcity_price = sorted_price[np.arange(n), cutoff_index][:, None]
        out = {}

        def add(name, values):
            values = np.broadcast_to(values, (n, HORIZON)).ravel().astype(np.float32)
            if not np.isfinite(values).all():
                raise ValueError(f'Non-finite feature {plant}/{name}')
            out['v3_' + name] = values

        add('connected_fill', initial / cap)
        add('connected_minus_local_fill', initial / cap - source_initial / source_capacity)
        add('connected_slack', (initial + cumulative - minimum) / cap)
        add('connected_remaining_inflow', remaining / cap)
        add('connected_inflow_to_rated_flow', flow / rated_flow)
        add('connected_available_full_load_hours', hours_budget)
        add('connected_mandatory_full_load_hours', required_hours)
        add('price_minus_storage_budget_cutoff', price - scarcity_price)
        add('storage_budget_minus_water_value', scarcity_price - terminal)
        add('head', head)
        add('initial_hours_at_rated_flow', initial / hourly_release)
        add('price_times_head', price * head)
        add('no_release_spill_fill', np.maximum(initial + cumulative - cap, 0) / cap)
        catchment_capacity = sum(float(reservoirs[s]['max_vol']) for s in catchment)
        catchment_volume = volume[catchment].sum(axis=1).to_numpy()[:, None]
        catchment_flow_columns = catchment + [r for r in rivers if r in inflow.columns]
        catchment_flow = inflow[catchment_flow_columns].sum(axis=1).to_numpy().reshape(n, HORIZON)
        catchment_water = sum(water[s].to_numpy()[:, None] * reservoirs[s]['max_vol'] for s in catchment) / catchment_capacity
        mandatory_flow = sum((min_flow[r].to_numpy().reshape(n, HORIZON) for r in rivers if r in min_flow.columns), np.zeros_like(price))
        add('catchment_fill', catchment_volume / catchment_capacity)
        add('catchment_inflow_to_rated', catchment_flow / rated_flow)
        add('catchment_inflow_week_to_rated', catchment_flow.mean(axis=1)[:, None] / rated_flow)
        add('catchment_remaining_inflow_fill', np.cumsum(catchment_flow[:, ::-1] * .0036, axis=1)[:, ::-1] / catchment_capacity)
        add('catchment_minus_local_water', catchment_water - terminal)
        add('catchment_mandatory_flow_to_rated', mandatory_flow / rated_flow)
        add('catchment_initial_full_load_hours', catchment_volume / hourly_release)
        add('catchment_no_release_excess', np.maximum(catchment_volume + np.cumsum(catchment_flow*.0036, axis=1) - catchment_capacity, 0) / catchment_capacity)
        for reservoir in downstream:
            add('downstream_fill_' + reservoir, volume[reservoir].to_numpy()[:, None] / reservoirs[reservoir]['max_vol'])
            add('local_minus_downstream_water_' + reservoir, terminal - water[reservoir].to_numpy()[:, None])
        margin = price - terminal
        # A local EUR/MWh water value is already relative to the downstream plant;
        # do not incorrectly subtract a downstream EUR/MWh value a second time.
        for threshold_name, threshold in [('water', terminal), ('budget', np.maximum(terminal, scarcity_price))]:
            gain = price - threshold
            reward = np.where(gain >= 0, gain, gain * pmin_fraction)
            for scale in (0.5, 1.0, 2.0):
                advantage = commitment_advantage(reward, startup * scale, shutdown * scale)
                add(f'dp_{threshold_name}_{scale:g}', advantage)
                add(f'dp_{threshold_name}_{scale:g}_on', advantage > 0)
            dispatch_proxy = np.maximum(gain, 0) > 0
            release = np.cumsum(dispatch_proxy * hourly_release, axis=1)
            add(f'proxy_storage_{threshold_name}', (initial + cumulative - release) / cap)
        for width in (3, 12, 48, 96, 168):
            positive = window_sum(np.maximum(margin, 0), width)
            negative = window_sum(np.maximum(-margin, 0), width)
            add(f'profit_next_{width}', positive)
            add(f'loss_next_{width}', negative)
            add(f'profit_past_{width}', window_sum(np.maximum(margin, 0), width, False))
        days = price.reshape(n, 7, 24)
        for statistic, values in [('min', days.min(axis=2)), ('max', days.max(axis=2)), ('std', days.std(axis=2))]:
            add('daily_' + statistic, np.repeat(values, 24, axis=1))
        for q in (.1, .25, .5, .75, .9):
            add(f'margin_quantile_{q}', np.quantile(price, q, axis=1)[:, None] - terminal)
        plants[plant] = pd.DataFrame(out)
        audit.append(dict(plant=plant, direct_sources='|'.join(sources), connected_storage='|'.join(group), passive_catchment='|'.join(catchment),
                          downstream='|'.join(downstream), total_MW=pmax, start_EUR_per_MW=startup))
    return plants, pd.DataFrame(audit)

## Definitions: runner (no training)

In [5]:
"""Sequential, resumable challenger. Importing this module never trains models."""
import gc
import hashlib
import importlib.metadata
import json
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits


def raw_digest(root):
    paths = sorted((root / 'data').rglob('*.csv')) + sorted((root / 'data').rglob('*.yaml'))
    paths += [root / 'prediction_mapping.csv', root / 'sample_submission.csv']
    digest = hashlib.sha256()
    for path in paths:
        digest.update(str(path.relative_to(root)).encode())
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
    return digest.hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix('.tmp.json')
    temporary.write_text(json.dumps(value, indent=2), encoding='utf-8')
    os.replace(temporary, path)


def purged_year(data, year):
    starts = pd.DatetimeIndex(data.train_cases['starttime'])
    begin = pd.Timestamp(f'{year}-01-01', tz='UTC')
    end = pd.Timestamp(f'{year+1}-01-01', tz='UTC')
    train = np.flatnonzero(starts + pd.Timedelta(hours=HORIZON-1) < begin)
    valid = np.flatnonzero((starts >= begin) & (starts < end))
    if not len(train) or not len(valid):
        raise ValueError('Empty temporal split')
    assert starts[train].max() + pd.Timedelta(hours=HORIZON-1) < starts[valid].min()
    return train, valid


def score_cube(truth, prediction):
    return float(roc_auc_score(truth.ravel(), prediction.ravel()))


def fit_member(spec, X, y, rounds, threads, valid=None):
    parameters = dict(spec['parameters'])
    if spec['backend'] == 'xgboost':
        from xgboost import XGBClassifier
        model = XGBClassifier(**parameters, n_estimators=int(rounds), n_jobs=threads,
                              **({'early_stopping_rounds': 80} if valid is not None else {}))
        model.fit(X, y, eval_set=[valid] if valid is not None else None, verbose=False)
        best = int(model.best_iteration) + 1 if valid is not None else int(rounds)
    else:
        import lightgbm as lgb
        parameters['num_threads'] = threads
        training = lgb.Dataset(X, label=y, free_raw_data=True)
        options = {}
        if valid is not None:
            parameters['metric'] = 'auc'
            options = dict(valid_sets=[lgb.Dataset(valid[0], label=valid[1], reference=training)],
                           callbacks=[lgb.early_stopping(80, verbose=False)])
        model = lgb.train(parameters, training, num_boost_round=int(rounds), **options)
        best = int(model.best_iteration or rounds)
    return model, best


class ChallengerRunner:
    def __init__(self, data, extra, local, additions, specs, threads, source_digest):
        self.data, self.local, self.additions = data, local, additions
        self.specs, self.threads = specs, min(4, int(threads))
        self.columns = list(data.feature_sets['full_engineered']) + list(extra.columns)
        self.base = np.column_stack([data.X_all[data.feature_sets['full_engineered']].to_numpy(np.float32),
                                     extra.to_numpy(np.float32)])
        self.raw = raw_digest(data.root)
        self.source_digest = source_digest
        self.directory = data.root / 'artifacts/challenger_v3'
        self.directory.mkdir(parents=True, exist_ok=True)

    def run_stage(self, member, train, predict, stage, rounds=None, tune=False):
        spec = self.specs[member]
        manifest = dict(member=member, spec=spec, train=list(map(int, train)),
                        predict=list(map(int, predict)), stage=stage, rounds=rounds,
                        tune=tune, source=self.source_digest, raw=self.raw, threads=self.threads,
                        packages={p: importlib.metadata.version(p) for p in ('numpy', 'pandas', 'scikit-learn', spec['backend'])})
        signature = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:20]
        directory = self.directory / member / signature
        directory.mkdir(parents=True, exist_ok=True)
        save_json(directory / 'manifest.json', manifest)
        cube = np.empty((len(predict), len(self.data.units), HORIZON), np.float32)
        selected, timings = {}, []
        train_rows, pred_rows = rows_for_cases(train), rows_for_cases(predict)
        for i, unit in enumerate(self.data.units):
            start = time.perf_counter()
            checkpoint = directory / f'unit_{i:02d}.npz'
            if checkpoint.exists():
                with np.load(checkpoint, allow_pickle=False) as saved:
                    values = saved['prediction'].copy()
                    best = int(saved['rounds'])
                status = 'restored'
            else:
                plant = unit.rsplit('_G', 1)[0]
                parts = [self.base, self.local[plant].to_numpy(np.float32)]
                names = self.columns + list(self.local[plant].columns)
                if spec['features'] == 'dispatch':
                    parts.append(self.additions[plant].to_numpy(np.float32))
                    names += list(self.additions[plant].columns)
                X = np.ascontiguousarray(np.column_stack(parts), dtype=np.float32)
                del parts
                y = self.data.target_cube[train, i].ravel()
                count = int(rounds[unit]) if rounds is not None else int(spec['max_rounds'])
                print(f'{stage} | {member} | {unit} | up to {count} rounds', flush=True)
                if len(np.unique(y)) == 1:
                    values = np.full((len(predict), HORIZON), y[0], np.float32)
                    best = 1
                else:
                    validation = (X[pred_rows], self.data.target_cube[predict, i].ravel()) if tune else None
                    # Limit BLAS as well as each model's own OpenMP threads.
                    with threadpool_limits(limits=self.threads), threadpool_limits(limits=1, user_api='blas'):
                        model, best = fit_member(spec, X[train_rows], y, count, self.threads, validation)
                        if spec['backend'] == 'xgboost':
                            values = model.predict_proba(X[pred_rows])[:, 1]
                            gain = model.feature_importances_
                        else:
                            values = model.predict(X[pred_rows], num_threads=self.threads)
                            gain = model.feature_importance(importance_type='gain')
                        values = values.reshape(len(predict), HORIZON).astype(np.float32)
                    pd.DataFrame({'feature': names, 'gain': gain}).sort_values('gain', ascending=False).to_csv(
                        directory / f'importance_{i:02d}.csv', index=False)
                    del model, validation
                temporary = checkpoint.with_suffix('.tmp.npz')
                np.savez_compressed(temporary, prediction=values, rounds=best)
                os.replace(temporary, checkpoint)
                del X, y
                gc.collect()
                status = 'trained'
            if values.shape != (len(predict), HORIZON) or not np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
                raise ValueError(f'Invalid predictions: {checkpoint}')
            cube[:, i] = values
            selected[unit] = int(best)
            seconds = time.perf_counter() - start
            timings.append(dict(unit=unit, rounds=best, seconds=seconds, status=status))
            pd.DataFrame(timings).to_csv(directory / 'runtime.csv', index=False)
            print(f'  {status}: {best} rounds, {seconds:.1f}s ({i+1}/{len(self.data.units)})', flush=True)
        return cube, selected

    def final_predictions(self, selection):
        train = np.arange(self.data.n_train_cases)
        test = np.arange(self.data.n_train_cases, self.data.n_all_cases)
        result = None
        for member, weight in selection['weights'].items():
            if weight <= 0:
                continue
            values, _ = self.run_stage(member, train, test, 'final', selection['rounds'][member])
            if result is None:
                result = np.zeros_like(values)
            result += float(weight) * values
        if result is None or abs(sum(selection['weights'].values()) - 1) > 1e-8:
            raise ValueError('Invalid ensemble weights')
        return np.clip(result, 0, 1)


def screen_challenger(runner, baseline_rounds):
    """2021 development, 2022 historical audit; final test labels are unavailable.

    These years have already informed earlier experiments, so the audit is not
    a new untouched test set. No per-unit weight search is performed.
    """
    predictions, all_rounds, rows = {}, {}, []
    for year in (2021, 2022):
        train, valid = purged_year(runner.data, year)
        predictions[year] = {}
        for member, spec in runner.specs.items():
            tuning = year == 2021 and spec['features'] == 'dispatch'
            counts = None if tuning else baseline_rounds.get(member, all_rounds.get(member))
            p, selected = runner.run_stage(member, train, valid, f'validation_{year}', counts, tune=tuning)
            all_rounds[member] = selected
            predictions[year][member] = p
            score = score_cube(runner.data.target_cube[valid], p)
            rows.append(dict(year=year, model=member, micro_auc=score))
            print(f'{year} | {member}: micro AUC={score:.6f}', flush=True)
        baseline = 0.5 * predictions[year]['xgb_v2'] + 0.5 * predictions[year]['lgb_v2']
        challenger = 0.5 * predictions[year]['xgb_dispatch'] + 0.5 * predictions[year]['lgb_dispatch']
        predictions[year]['baseline'] = baseline
        predictions[year]['challenger'] = challenger
        for label, p in [('baseline_50_50', baseline), ('dispatch_50_50', challenger)]:
            rows.append(dict(year=year, model=label, micro_auc=score_cube(runner.data.target_cube[valid], p)))
    scores = pd.DataFrame(rows)
    scores.to_csv(runner.directory / 'validation_scores.csv', index=False)
    _, development = purged_year(runner.data, 2021)
    _, audit = purged_year(runner.data, 2022)
    grid = []
    for alpha in (0., .25, .5, .75, 1.):
        p = (1-alpha)*predictions[2021]['baseline'] + alpha*predictions[2021]['challenger']
        grid.append((score_cube(runner.data.target_cube[development], p), alpha))
    # A tie favours the simpler existing baseline. Weights are global for micro AUC.
    dev_score, alpha = max(grid, key=lambda pair: (pair[0], -pair[1]))
    audited = (1-alpha)*predictions[2022]['baseline'] + alpha*predictions[2022]['challenger']
    audit_score = score_cube(runner.data.target_cube[audit], audited)
    baseline_score = score_cube(runner.data.target_cube[audit], predictions[2022]['baseline'])
    accepted = alpha > 0 and audit_score > baseline_score
    selected_alpha = alpha if accepted else 0.
    selection = dict(
        weights={'xgb_v2': (1-selected_alpha)/2, 'lgb_v2': (1-selected_alpha)/2,
                 'xgb_dispatch': selected_alpha/2, 'lgb_dispatch': selected_alpha/2},
        rounds=all_rounds, development_alpha=alpha, audit_candidate_auc=audit_score,
        audit_baseline_auc=baseline_score, passed_audit=bool(accepted),
        source_digest=runner.source_digest, raw_digest=runner.raw, specs=runner.specs)
    save_json(runner.directory / 'selection.json', selection)
    print('2021-selected challenger weight:', alpha, flush=True)
    print(f'2022 audit: candidate {audit_score:.6f}, baseline {baseline_score:.6f}', flush=True)
    print('Accepted new features.' if accepted else 'No confirmed improvement: guarded output uses baseline.', flush=True)
    # Diagnostics are descriptive only; these do not tune per-unit weights.
    details = []
    for i, unit in enumerate(runner.data.units):
        truth = runner.data.target_cube[audit, i].ravel()
        if len(np.unique(truth)) < 2:
            continue
        for name in ('baseline', 'challenger'):
            details.append(dict(unit=unit, model=name,
                                auc=float(roc_auc_score(truth, predictions[2022][name][:, i].ravel()))))
    pd.DataFrame(details).to_csv(runner.directory / 'audit_unit_scores.csv', index=False)
    return selection, scores

## Build features and inspect the storage mapping (no training)

In [6]:
data = build_project_data(ROOT)
extra, local, original_mapping = build_research_features(data)
dispatch, storage_mapping = build_challenger_features(data)
runner = ChallengerRunner(data, extra, local, dispatch, SPECS, THREADS, SOURCE_DIGEST)
print('Cases:', data.n_train_cases, 'train;', data.n_test_cases, 'test')
print('Extra dispatch features per plant:', {p: len(v.columns) for p, v in dispatch.items()})
print('Resident feature arrays, excluding temporary training copies (GB):', round(
    (runner.base.nbytes + sum(v.memory_usage(index=False).sum() for v in local.values())
     + sum(v.memory_usage(index=False).sum() for v in dispatch.values())) / 1e9, 2))
display(storage_mapping)
storage_mapping.to_csv(runner.directory / 'storage_feature_mapping.csv', index=False)

Cases: 2922 train; 725 test
Extra dispatch features per plant: {'Hogga': 58, 'Tokke': 60, 'Lio': 58, 'Byrte': 60, 'Vinje': 60, 'Haukeli': 60, 'Songa': 60, 'Kjela': 60, 'Vesle Kjela': 60}
Resident feature arrays, excluding temporary training copies (GB): 2.32


,plant,direct_sources,connected_storage,passive_catchment,downstream,total_MW,start_EUR_per_MW
0,Hogga,Bandak,Bandak,Bandak|Bitdalsvatn|Bordalsvatn|Botnedalsvatn|B...,,17.0,5.529412
1,Tokke,Vinjevatn,Vinjevatn,Bitdalsvatn|Bordalsvatn|Foersvatn|Hyljelihyl|K...,Bandak,430.0,1.990698
2,Lio,Byrtevatn,Byrtevatn,Byrtevatn,,43.0,3.790698
3,Byrte,Botnedalsvatn,Botnedalsvatn,Botnedalsvatn,Byrtevatn,20.0,6.750000
4,Vinje,Vaamarvatn,Totak|Vaamarvatn|Venemo,Bitdalsvatn|Bordalsvatn|Foersvatn|Hyljelihyl|K...,Vinjevatn,300.0,1.980000
5,Haukeli,Vatjern,Vatjern,Langeidvatn|Vatjern,Vinjevatn,4.9,29.183673
6,Songa,Bitdalsvatn|Songavatn,Bitdalsvatn|Songavatn,Bitdalsvatn|Songavatn,Totak,120.0,1.850000
7,Kjela,Foersvatn,Bordalsvatn|Foersvatn,Bordalsvatn|Foersvatn|Langesae,Hyljelihyl,60.0,2.850000
8,Vesle Kjela,Kjelavatn,Kjelavatn,Kjelavatn|Staavatn,Foersvatn,8.5,10.823529


## TRAINING A — historical validation

In [7]:
# TRAINING A — development and historical audit. Completed unit tasks resume.
selection, validation_scores = screen_challenger(runner, BASELINE_ROUNDS)
display(validation_scores.pivot(index='model', columns='year', values='micro_auc'))
print('Recommended final weights:', selection['weights'])
print('Historical audit accepted the challenger:', selection['passed_audit'])

validation_2021 | xgb_v2 | Hogga_G1 | up to 410 rounds
  trained: 410 rounds, 20.4s (1/14)
validation_2021 | xgb_v2 | Tokke_G1 | up to 450 rounds
  trained: 450 rounds, 24.5s (2/14)
validation_2021 | xgb_v2 | Tokke_G2 | up to 854 rounds
  trained: 854 rounds, 40.6s (3/14)
validation_2021 | xgb_v2 | Tokke_G3 | up to 587 rounds
  trained: 587 rounds, 30.1s (4/14)
validation_2021 | xgb_v2 | Tokke_G4 | up to 693 rounds
  trained: 693 rounds, 32.4s (5/14)
validation_2021 | xgb_v2 | Lio_G1 | up to 249 rounds
  trained: 249 rounds, 12.0s (6/14)
validation_2021 | xgb_v2 | Byrte_G1 | up to 292 rounds
  trained: 292 rounds, 14.1s (7/14)
validation_2021 | xgb_v2 | Vinje_G1 | up to 658 rounds
  trained: 658 rounds, 32.6s (8/14)
validation_2021 | xgb_v2 | Vinje_G2 | up to 404 rounds
  trained: 404 rounds, 21.2s (9/14)
validation_2021 | xgb_v2 | Vinje_G3 | up to 607 rounds
  trained: 607 rounds, 28.1s (10/14)
validation_2021 | xgb_v2 | Haukeli_G1 | up to 223 rounds
  trained: 223 rounds, 12.4s (11/1

year,2021,2022
model,,
baseline_50_50,0.984398,0.986219
dispatch_50_50,0.979461,0.980326
lgb_dispatch,0.984384,0.984336
lgb_v2,0.983760,0.986065
xgb_dispatch,0.973625,0.975513
xgb_v2,0.984074,0.985523


Recommended final weights: {'xgb_v2': 0.5, 'lgb_v2': 0.5, 'xgb_dispatch': 0.0, 'lgb_dispatch': 0.0}
Historical audit accepted the challenger: False


## TRAINING B — full-history predictions

In [9]:
# TRAINING B — full-history fits, with already chosen rounds and weights.
# After a restart: rerun configuration/definitions/data, then load the saved selection.
if 'selection' not in globals():
    selection_path = runner.directory / 'selection.json'
    if not selection_path.exists():
        raise FileNotFoundError('Run TRAINING A before TRAINING B.')
    selection = json.loads(selection_path.read_text())
    if (selection.get('source_digest') != SOURCE_DIGEST or selection.get('raw_digest') != runner.raw
            or selection.get('specs') != SPECS):
        raise ValueError('Saved selection belongs to changed code/data. Rerun TRAINING A.')
guarded = runner.final_predictions(selection)
submission_path, submission = make_submission(data, guarded, 'hydro_dispatch_v3.csv')
print('Recommended CSV:', submission_path)
print('Passed historical audit:', selection['passed_audit'])
# Export a separately named pure challenger to make the experiment inspectable.
experimental_selection = dict(selection, weights={'xgb_dispatch': .5, 'lgb_dispatch': .5})
experimental = runner.final_predictions(experimental_selection)
experimental_path, _ = make_submission(data, experimental, 'hydro_dispatch_experimental.csv')
print('Pure challenger (inspect validation before upload):', experimental_path)
print(f'Total elapsed including features and validation: {(time.perf_counter()-NOTEBOOK_STARTED)/60:.1f} minutes')
display(submission.iloc[:3, :8])

  restored: 410 rounds, 0.0s (1/14)
  restored: 450 rounds, 0.0s (2/14)
final | xgb_v2 | Tokke_G2 | up to 854 rounds
  trained: 854 rounds, 55.0s (3/14)
final | xgb_v2 | Tokke_G3 | up to 587 rounds
  trained: 587 rounds, 40.5s (4/14)
final | xgb_v2 | Tokke_G4 | up to 693 rounds
  trained: 693 rounds, 43.7s (5/14)
final | xgb_v2 | Lio_G1 | up to 249 rounds
  trained: 249 rounds, 16.0s (6/14)
final | xgb_v2 | Byrte_G1 | up to 292 rounds
  trained: 292 rounds, 18.4s (7/14)
final | xgb_v2 | Vinje_G1 | up to 658 rounds
  trained: 658 rounds, 42.5s (8/14)
final | xgb_v2 | Vinje_G2 | up to 404 rounds
  trained: 404 rounds, 27.6s (9/14)
final | xgb_v2 | Vinje_G3 | up to 607 rounds
  trained: 607 rounds, 37.5s (10/14)
final | xgb_v2 | Haukeli_G1 | up to 223 rounds
  trained: 223 rounds, 16.5s (11/14)
final | xgb_v2 | Songa_G1 | up to 510 rounds
  trained: 510 rounds, 34.8s (12/14)
final | xgb_v2 | Kjela_G1 | up to 326 rounds
  trained: 326 rounds, 22.6s (13/14)
final | xgb_v2 | Vesle Kjela_G1 |

,Run No,result_committed_Hogga_G1_t0,result_committed_Hogga_G1_t1,result_committed_Hogga_G1_t2,result_committed_Hogga_G1_t3,result_committed_Hogga_G1_t4,result_committed_Hogga_G1_t5,result_committed_Hogga_G1_t6
0,2923,0.108930,0.068554,0.039475,0.029846,0.020315,0.017901,0.008322
1,2924,0.868713,0.866741,0.885165,0.884662,0.884848,0.909455,0.925478
2,2925,0.999437,0.999460,0.999455,0.998925,0.999478,0.999556,0.999612


## Export short notebooks

In [10]:
# Generate self-contained final-only notebooks with the chosen constants embedded.
# The generated notebooks do not read this notebook, src/, caches, or input submissions.
source_path = ROOT / 'notebooks/16_hydro_dispatch_challenger.ipynb'
if not source_path.is_file():
    raise FileNotFoundError('Keep notebook 16 under notebooks/ while exporting the short versions.')
source_notebook = json.loads(source_path.read_text(encoding='utf-8'))

def export_short(chosen, filename, output_csv):
    keep = []
    for c in source_notebook['cells']:
        if c['id'] in ('config', 'pipeline', 'physics', 'dispatch', 'runner', 'build'):
            copied = json.loads(json.dumps(c))
            copied['execution_count'] = None
            copied['outputs'] = []
            if c['id'] == 'runner':
                text = ''.join(copied['source']).split('def screen_challenger(')[0]
                copied['source'] = text.splitlines(keepends=True)
            keep.append(copied)
    constants = 'selection = ' + repr(chosen) + '\n'
    execute = ('prediction = runner.final_predictions(selection)\n'
               + f'path, submission = make_submission(data, prediction, {output_csv!r})\n'
               + 'print("Saved:", path)\n'
               + 'print("Elapsed minutes:", (time.perf_counter()-NOTEBOOK_STARTED)/60)\n')
    # Only scalar hyperparameters, unit tree counts and blend weights are embedded.
    keep += [dict(cell_type='code', id='frozen-selection', metadata={}, execution_count=None, outputs=[], source=constants.splitlines(keepends=True)),
             dict(cell_type='code', id='final-fit', metadata={}, execution_count=None, outputs=[], source=execute.splitlines(keepends=True))]
    intro = ('# Short notebook — hydro dispatch ensemble\n\n'
             '**Henrik Remmen (541747), Group 119, [119] SINTEF Databreach**\n\n'
             'Offline raw-data reproduction; max four CPU threads. Run all cells. '
             'Counts/weights selected in notebook 16 and described in the research report. '
             'No precomputed predictions or trained models are required.\n')
    keep.insert(0, dict(cell_type='markdown', id='short-intro', metadata={}, source=intro.splitlines(keepends=True)))
    result = dict(source_notebook, cells=keep)
    destination = runner.directory / filename
    destination.write_text(json.dumps(result, indent=1, ensure_ascii=False)+'\n', encoding='utf-8')
    print('Final-only notebook:', destination)

export_short(selection, 'Short_notebook_dispatch_v3.ipynb', 'hydro_dispatch_v3.csv')
export_short(experimental_selection, 'Short_notebook_dispatch_experimental.ipynb', 'hydro_dispatch_experimental.csv')

Final-only notebook: /Users/henrik/Documents/H26/Moderne Maksinlæring/sintef_project/artifacts/challenger_v3/Short_notebook_dispatch_v3.ipynb
Final-only notebook: /Users/henrik/Documents/H26/Moderne Maksinlæring/sintef_project/artifacts/challenger_v3/Short_notebook_dispatch_experimental.ipynb


## Sources and interpretation

- [SINTEF SHOP water values](https://docs.shop.sintef.energy/examples/water_values/water_values.html): local EUR/MWh values are relative to the first downstream plant; SHOP converts these internally.
- [SINTEF SHOP generators](https://docs.shop.sintef.energy/objects/generator/generator.html): head, discharge, efficiency and start/stop economics motivate the proxy features.
- [LightGBM tuning](https://lightgbm.readthedocs.io/en/stable/Parameters-Tuning.html): leaves and minimum leaf size control capacity and cost.
- [XGBoost parameters](https://xgboost.readthedocs.io/en/stable/parameter.html): bounded histogram tree training and regularization.

These sources motivate hypotheses, not a prediction of Kaggle score. Only the supplied course data
are used. The features approximate hydraulic connectivity and do not enforce full hydraulic feasibility.
See `RESEARCH_DISPATCH_V3.md` for local error analysis and selection rationale.